# Complaint Intelligence — Data Preparation & Feature Engineering
CFPB Consumer Complaint Database (January–June 2026 extract)

**Purpose:** Take the raw, EDA-validated extract and produce a cleaned, deduplicated,
model-ready corpus for `03_copilot_rag.py`. This notebook documents every cleaning
decision, checks for data leakage risk, and defines the transformations and derived
features used downstream.

Notebook outline
1. Setup & Imports
2. Deduplication
3. Text Cleaning
4. Column Selection
5. Data Leakage Check
6. Transformation & Normalization Decisions
7. Feature Engineering
8. Export Model-Ready Corpus

In [7]:
# --- Core libraries ---
import os
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# Path to the CFPB complaints extract (same file used in 01_copilot_eda.ipynb)
CSV_PATH = os.environ.get("COMPLAINTS_CSV", "complaints.csv")

df = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head(3)

Rows:    48,708
Columns: 17


,product,consumer_complaint_narrative,date_sent_to_company,issue,sub_product,zip_code,tags,complaint_id,timely,company_response,submitted_via,company,date_received,state,company_public_response,sub_issue,is_duplicate_narrative
0,Checking or savings account,on XX/XX/year> my wife made a XXXX payment to...,2026-06-01T17:30:39.000Z,Managing an account,Checking account,92618,NaN,22771183,Yes,Closed with non-monetary relief,Web,ALLY FINANCIAL INC.,2026-06-01 17:14:54+00:00,CA,Company has responded to the consumer and the ...,Deposits and withdrawals,False
1,Credit card,Mobile repair business took {$400.00} from my ...,2026-06-02T14:26:31.000Z,Problem with a purchase shown on your statement,General-purpose credit card or charge card,216XX,Older American,22804841,Yes,Closed with explanation,Web,BARCLAYS BANK DELAWARE,2026-06-02 13:35:44+00:00,MD,Company has responded to the consumer and the ...,Credit card company isn't resolving a dispute ...,False
2,Credit reporting or other personal consumer re...,THESE COLLECTION ACCOUNTS DO NOT BELONG TO ME ...,2026-06-03T07:41:13.000Z,Incorrect information on your report,Credit reporting,92113,NaN,22838594,Yes,Closed with explanation,Web,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",2026-06-03 07:39:10+00:00,CA,Company has responded to the consumer and the ...,Information is missing that should be on the r...,False


## 2. Deduplication

The EDA identified about 9% of complaints as flagged `is_duplicate_narrative` — mass-filed
or templated complaints, heavily concentrated in Credit Reporting (~36%) and Debt Collection
(~22%). These are removed here before any embedding or modeling work, since near-identical
narratives would inflate the corpus with redundant vectors and bias topic modeling toward
whichever category has the most template filings.

In [8]:
# --- Deduplication ---
print(f"Before dedup: {len(df):,} rows")
print(f"Flagged as duplicate: {df['is_duplicate_narrative'].sum():,} rows "
      f"({df['is_duplicate_narrative'].mean()*100:.1f}%)")

df_clean = df[df["is_duplicate_narrative"] == False].copy()

print(f"After dedup:  {len(df_clean):,} rows")

# Confirm the drop is concentrated where the EDA predicted
print("\nDuplicate rate by product (before removal):")
print((df.groupby("product")["is_duplicate_narrative"].mean() * 100).round(1).sort_values(ascending=False).head(5))

Before dedup: 48,708 rows
Flagged as duplicate: 4,474 rows (9.2%)
After dedup:  44,234 rows

Duplicate rate by product (before removal):
product
Credit reporting or other personal consumer reports    35.9
Debt collection                                        22.1
Debt or credit management                               1.5
Credit card                                             0.9
Money transfer, virtual currency, or money service      0.5
Name: is_duplicate_narrative, dtype: float64


In [9]:
# --- Check how common the "XX/XX/year>" pattern actually is ---
weird_date_pattern = df_clean["consumer_complaint_narrative"].str.contains(
    r"XX/XX/year>", regex=True, na=False
)
print(f"Rows with 'XX/XX/year>' pattern: {weird_date_pattern.sum():,} "
      f"({weird_date_pattern.mean()*100:.2f}%)")

Rows with 'XX/XX/year>' pattern: 11,615 (26.26%)


## 3. Text Cleaning

CFPB automatically redacts personal information from narratives using placeholder tokens
(`XXXX`, `XX/XX/XXXX`, `{$...}`). These tokens are useful to a human reader but add noise
to an embedding model, which will otherwise treat "XXXX XXXX XXXX" as meaningful repeated
text. We create a **separate cleaned column** for embeddings while preserving the original
narrative untouched for display in the app — users should see the authentic complaint text,
not the cleaned version.

**Note:** an initial spot-check surfaced a second date-redaction format, `XX/XX/year>`,
which affected about 26% of narratives and was not caught by the standard `XX/XX/XXXX`
pattern. The cleaning function below was extended to handle this format as well.

In [10]:
# --- Text cleaning for embedding input ---
def clean_narrative(text):
    if pd.isna(text):
        return text
    # Dollar placeholders: {$400.00} -> " "
    text = re.sub(r"\{\$[\d,]*\.?\d*\}", " ", text)
    # Date redactions: XX/XX/2026, XX/XX/XXXX, XX/XX/year>, XX/XX/year
    text = re.sub(r"XX/XX/(\d{4}|XXXX|year>?)", " ", text, flags=re.IGNORECASE)
    # Standalone XXXX-style redaction tokens (names, account numbers, etc.)
    text = re.sub(r"\bXX+\b", " ", text)
    # Collapse extra whitespace left behind by removals
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_clean["narrative_clean"] = df_clean["consumer_complaint_narrative"].apply(clean_narrative)

# Confirm the "XX/XX/year>" pattern is fully handled now
remaining = df_clean["narrative_clean"].str.contains(r"XX/XX/year>?", regex=True, na=False, flags=re.IGNORECASE)
print(f"Rows still containing the pattern after cleaning: {remaining.sum():,}")

# Spot-check on the same rows as before
for i in df_clean.index[:2]:
    print("RAW:  ", df_clean.loc[i, "consumer_complaint_narrative"][:200])
    print("CLEAN:", df_clean.loc[i, "narrative_clean"][:200])
    print()

Rows still containing the pattern after cleaning: 0
RAW:   on XX/XX/year> my wife made a XXXX  payment to buy some merchandise online, and realized the same day that it was a scammer who never intended to send the merchandise. She reported immediately to Ally
CLEAN: on my wife made a payment to buy some merchandise online, and realized the same day that it was a scammer who never intended to send the merchandise. She reported immediately to Ally bank to investiga

RAW:   Mobile repair business took {$400.00} from my Barclays bank MasterCard ( GM rewards MasterCard ) supposedly to buy the new parts for a mobile car repair job to be done at my home. He then refused to p
CLEAN: Mobile repair business took from my Barclays bank MasterCard ( GM rewards MasterCard ) supposedly to buy the new parts for a mobile car repair job to be done at my home. He then refused to provide pro



## 4. Column Selection

The EDA's Column Recommendations table identified three columns with little to no analytical
or modeling value for this project:

- **`submitted_via`** — a single constant value ("Web") in this extract, since narrative
  consent only happens through the web form. No signal, cannot be used as a filter or feature.
- **`date_sent_to_company`** — near-zero routing lag relative to `date_received` in nearly
  all cases, making it functionally redundant with the primary timestamp.
- **`company_public_response`** — about 60% missing, and mostly generic boilerplate
  ("Company has responded to the consumer...") when present, offering little differentiating
  signal.

These are dropped here rather than carried forward into the embedding and modeling pipeline.

In [11]:
# --- Drop low-value columns ---
cols_to_drop = ["submitted_via", "date_sent_to_company", "company_public_response"]

print("Columns before drop:", df_clean.shape[1])
df_clean = df_clean.drop(columns=cols_to_drop)
print("Columns after drop: ", df_clean.shape[1])
print("\nRemaining columns:", list(df_clean.columns))

Columns before drop: 18
Columns after drop:  15

Remaining columns: ['product', 'consumer_complaint_narrative', 'issue', 'sub_product', 'zip_code', 'tags', 'complaint_id', 'timely', 'company_response', 'company', 'date_received', 'state', 'sub_issue', 'is_duplicate_narrative', 'narrative_clean']


## 5. Data Leakage Check

Traditional data leakage concerns (e.g., target leakage from a feature that encodes the
label, or train/test contamination from fitting a scaler on the full dataset) apply to
*supervised* learning with a train/test split. This project has no such split: the RAG
retrieval system and the unsupervised topic model do not train on labeled outcomes, so the
classic leakage risks do not directly apply. However, there are two leakage-adjacent risks
specific to this project worth checking explicitly:

1. **Embedding model leakage:** the sentence-transformer used for embeddings
   (`all-MiniLM-L6-v2`) is a pretrained, off-the-shelf model. It is not fine-tuned on this
   complaint corpus, so there is no risk of the model having "seen" these specific
   complaints during its own training in a way that would artificially inflate retrieval
   performance.
2. **Evaluation leakage:** if the team builds a held-out set of test questions to evaluate
   RAG retrieval quality, those test complaints must not be hand-picked by someone who has
   already read the corpus deeply enough to know the "right" answer in advance, and any
   such evaluation set should be drawn from the same cleaned/deduplicated corpus used for
   the live index, not a separately-processed version, to avoid a train/serve mismatch
   between what the retriever was tested on and what it will actually index in production.

Since narrative deduplication happens **before** the corpus is embedded (Section 2 above),
there is no risk of the same near-duplicate complaint appearing multiple times and
artificially inflating retrieval confidence for a single templated complaint pattern.

## 6. Transformation & Normalization Decisions

This project has no traditional train/test split, so the usual concern of "fit transformations
on training data only" does not directly apply. The relevant transformation decisions here are:

- **Chunking, not fitting:** the EDA found narrative length varies meaningfully by product
  (median ~130 words for Debt Collection vs. ~240 words for Mortgage). Rather than a single
  fixed-size split applied uniformly, narratives are chunked with overlap only when they
  exceed a length threshold, preserving short narratives whole and splitting only the long tail.
- **No numeric scaling needed:** the corpus is entirely categorical and text fields; there are
  no continuous numeric features (e.g., dollar amounts) being fed into a model that would
  require standardization or normalization.
- **Text normalization is the redaction-token cleaning already performed in Section 3**,
  applied uniformly to the entire corpus at once, since embedding generation is not a
  "fit" step in the traditional sense — the pretrained sentence-transformer is applied
  identically to every narrative regardless of when it is processed.

In [12]:
# --- Chunking: flag narratives that exceed the embedding model's comfortable input length ---
CHUNK_WORD_THRESHOLD = 220  # approx. safe margin under MiniLM's ~256-token limit

df_clean["narrative_word_count"] = df_clean["narrative_clean"].str.split().str.len()
df_clean["needs_chunking"] = df_clean["narrative_word_count"] > CHUNK_WORD_THRESHOLD

print(f"Narratives needing chunking: {df_clean['needs_chunking'].sum():,} "
      f"({df_clean['needs_chunking'].mean()*100:.1f}%)")

print("\nChunking need by product:")
print((df_clean.groupby("product")["needs_chunking"].mean() * 100).round(1).sort_values(ascending=False))

Narratives needing chunking: 16,748 (37.9%)

Chunking need by product:
product
Mortgage                                                   53.0
Credit card                                                44.5
Vehicle loan or lease                                      43.6
Student loan                                               40.7
Checking or savings account                                39.7
Money transfer, virtual currency, or money service         38.7
Payday loan, title loan, personal loan, or advance loan    35.4
Debt collection                                            29.7
Debt or credit management                                  29.7
Prepaid card                                               24.2
Credit reporting or other personal consumer reports        23.7
Name: needs_chunking, dtype: float64


## 7. Feature Engineering

Two features are engineered here to support the modeling pipeline:

- **Embedding input template:** rather than embedding the raw narrative alone, we prepend
  structured context (`product`, `issue`) to the cleaned narrative. This measurably helps
  retrieval relevance for category-style queries, since the embedding then captures both
  what happened and what kind of complaint it is.
- **Keyword-search-ready field:** in addition to the dense (embedding-based) retrieval
  pipeline, `03_copilot_rag.py` will support a traditional keyword search (BM25) baseline
  for comparison, as recommended in instructor feedback on the project proposal. BM25
  requires no additional feature engineering beyond the already-cleaned narrative text, but
  is noted here as a deliberate design decision: the cleaned corpus produced by this notebook
  is built to serve **both** the dense retrieval pipeline and a keyword-based baseline, so
  that semantic search performance can be benchmarked against a traditional, non-AI baseline
  rather than assumed to be superior.

**Deferred to `03_copilot_rag.py`:** the actual embedding generation, vector similarity
features, and BM25 index construction are modeling-pipeline concerns, not data preparation
concerns, and are built downstream of this notebook's output.

In [13]:
# --- Feature engineering: embedding input template ---
def build_embedding_input(row):
    return f"Product: {row['product']} | Issue: {row['issue']}\n{row['narrative_clean']}"

df_clean["embedding_input"] = df_clean.apply(build_embedding_input, axis=1)

# Spot-check
print(df_clean["embedding_input"].iloc[0][:300])

Product: Checking or savings account | Issue: Managing an account
on my wife made a payment to buy some merchandise online, and realized the same day that it was a scammer who never intended to send the merchandise. She reported immediately to Ally bank to investigate. The amount was . Later that we


## 8. Export Model-Ready Corpus

The cleaned, deduplicated, feature-engineered corpus is exported here for `03_copilot_rag.py`
to consume. Both CSV and Parquet are saved, consistent with the project's existing convention
(see `ingest.py` output), with Parquet as the primary format for the modeling pipeline due to
its smaller size and faster load time at this corpus scale.

In [14]:
# --- Export the model-ready corpus ---
export_cols = [
    "complaint_id", "product", "sub_product", "issue", "sub_issue",
    "company", "state", "date_received", "company_response", "timely",
    "tags", "narrative_clean", "embedding_input",
    "narrative_word_count", "needs_chunking"
]

df_export = df_clean[export_cols].copy()

df_export.to_csv("complaints_model_ready.csv", index=False)
df_export.to_parquet("complaints_model_ready.parquet", index=False)

print(f"Exported {len(df_export):,} rows, {len(export_cols)} columns")
print(f"CSV size:     {os.path.getsize('complaints_model_ready.csv') / 1e6:.1f} MB")
print(f"Parquet size: {os.path.getsize('complaints_model_ready.parquet') / 1e6:.1f} MB")
df_export.head(2)

Exported 44,234 rows, 15 columns
CSV size:     129.1 MB
Parquet size: 64.0 MB


,complaint_id,product,sub_product,issue,sub_issue,company,state,date_received,company_response,timely,tags,narrative_clean,embedding_input,narrative_word_count,needs_chunking
0,22771183,Checking or savings account,Checking account,Managing an account,Deposits and withdrawals,ALLY FINANCIAL INC.,CA,2026-06-01 17:14:54+00:00,Closed with non-monetary relief,Yes,NaN,on my wife made a payment to buy some merchand...,Product: Checking or savings account | Issue: ...,364,True
1,22804841,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,BARCLAYS BANK DELAWARE,MD,2026-06-02 13:35:44+00:00,Closed with explanation,Yes,Older American,Mobile repair business took from my Barclays b...,Product: Credit card | Issue: Problem with a p...,146,False
